In [1]:
import os
import datetime
# from netCDF4 import Dataset
import numpy as np
import pandas as pd
import xarray as xr
from scipy import signal, integrate, stats
import yaml
import importlib
import multiprocessing as mp
import time

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib
from matplotlib.colors import BoundaryNorm, LogNorm
from matplotlib.ticker import MultipleLocator, FuncFormatter, AutoMinorLocator
from matplotlib.ticker import MaxNLocator
from mpl_toolkits.mplot3d import Axes3D, axes3d

import cmaps, plt_helper, filter

plt.style.use('latex_default.mplstyle')

In [2]:
%matplotlib inline
plt.ioff()

# flavor = "pmap_mpdata"
flavor = "pmap_mpdata"
# flavor = "pmap_2N"
# flavor = "pmap_amp100"
# flavor = "pmap_bcy1"
# flavor = "pmap_circle"
# flavor = "pmap_1p5dt"
# flavor = "pmap_ppm"
# flavor = "eulag_compressi"
flavor = "eulag_anelastic"
flavor = "pmap_ld4"
flavor = "pmap_ld4px"

folder = "/work/bd0620/b309199/linear-mws"
sim = f"{flavor}_MW_010km_ramp"
# sim = f"{flavor}_MW_010km_cos"
sim = f"{flavor}_MW_010km_relax0"
sim = f"{flavor}_MW_010km_filw"
sim = f"{flavor}_MW_100km"
# sim = f"{flavor}_MW_010km"

model = flavor.split("_")[0]
fpath = os.path.join(folder, sim)

image_folder = f"../data/pmap-animations/{sim}"
os.makedirs(image_folder,exist_ok=True)

# var = "thprime"
# clev, clev_l = plt_helper.get_colormap_bins_and_labels(max_level=0.8)
# cbar_label = r"$\Theta'$ / K"

# var = "pprime"
# var = "thprime"
var = "w"
clev_factor = 10**(-2)
clev, clev_l = plt_helper.get_colormap_bins_and_labels(max_level=0.2)
# cbar_label = r"w' / m$\,$s$^{-1}$"
# cbar_label = r"$\Theta$' / K"
clev = [x * clev_factor for x in clev]
clev_l = [-0.4,-0.1,-0.025,0.025,0.1,0.4]
clev_l = [x * clev_factor for x in clev_l]
# clev   = [-32,-16,-8,-4,-2,-1,-0.5,0.5,1,2,4,8,16,32]
# clev_l = [-16,-4,-1,1,4,16]
cmap = cmaps.get_wave_cmap()
norm = BoundaryNorm(boundaries=clev , ncolors=cmap.N, clip=True)

z=0

def plot_mw_sim(t, fpath, pbar=None, sema=None):
    gskw  = {'hspace':0.05, 'wspace':0.03, 'width_ratios': [6,1]}
    fig, axes = plt.subplots(2,2, sharex=True, figsize=(9,8.5), gridspec_kw=gskw)
    ax0 = axes[0,0]
    ax1 = axes[1,0]
    axes[0,1].axis('off')
    axes[1,1].axis('off')

    if model == "pmap":
        dsxz, dsxy, cfg = plt_helper.preprocess_pmap(fpath, t=t, slices={"x": 0, "y": 0, "z": 0})
        dsxz['xcr']=dsxz.x.expand_dims(ycr=np.shape(dsxy[var])[0]) / 1000
        dsxz['zcr']=dsxz.zcr / 1000
        dsxy['xcr']=dsxy.x.expand_dims(y=np.shape(dsxy[var])[0])  / 1000
        dsxy['ycr']=dsxy.y.expand_dims(x=np.shape(dsxy[var])[1], axis=1)  / 1000
        dsxy['zcr']=dsxy['zcr'] / 1000
    else:
        _, dsxz, _, ds_xyslices = plt_helper.preprocess_eulag_tstep(fpath, t, slices={"x": 0, "y": 0, "z": [0]})
        dsxy = ds_xyslices[0]
        dsxy['zcr']=dsxy['zcrtopo'] / 1000
        cfg = None

    # ----- XZ cross-section -----
    contf0 = ax0.contourf(
        dsxz.xcr[0,:].expand_dims(z=np.shape(dsxz[var])[0]),
        dsxz.zcr[:,:],
        dsxz[var][:,:],
        cmap=cmap, norm=norm, levels=clev, extend='both'
    )
    ax0.plot(dsxz.xcr[0,:], 100*dsxz.zcr[0,:], lw=2, color='black')

    ax0.xaxis.set_label_position('top')
    ax0.set_ylabel("altitude z / km")
    ax0.set_xlabel("streamwise x / km")
    ax0.tick_params(which='both', labeltop=True)
    ax0.text(0.97, 0.9, 'timestep: {:03d}'.format(t), horizontalalignment='right', transform=ax0.transAxes, bbox={"boxstyle" : "round", "lw":0.67, "facecolor":"white", "edgecolor":"black"})

    # --- profile: add a twin-x axis that shares y (z) and plot ue(z) at x=0 ---
    ax_prof = ax0.twiny()                                 # shares y with ax0
    # move the profile axis to the bottom so it doesn't clash with the top x label
    ax_prof.spines["bottom"].set_position(("outward", 0))
    ax_prof.xaxis.set_label_position('bottom')
    ax_prof.xaxis.tick_bottom()
    ax_prof.set_xlim([0,60])
    ax_prof.set_xticks([10])
    ax_prof.tick_params(axis='x', pad=-12) 
    ax_prof.xaxis.set_major_formatter(
        FuncFormatter(lambda val, pos: rf"{val:g} " + r"$\mathrm{m\,s^{-1}}$")
    )
    ax0.xaxis.tick_top()

    # build the vertical profile (z vs ue at x=0)
    # z coordinate: take the first x-column so it's 1D
    z_km = (dsxz.zcr[:,0]).values
    if 'ue' in dsxz.variables:
        ue_prof = dsxz["ue"][:, 0].values
    else:
        ue_prof = dsxz["u"][:, 0].values
        # raise KeyError("Couldn't find 'ue' in dsxz; ensure the dataset has variable 'ue' with dims (z, x).") from e

    ax_prof.plot(ue_prof, z_km, lw=2.5, ls="--", color="black")                     # no color specified; uses default
    # ax_prof.set_xlabel(r"$u_e$ (m s$^{-1}$)")
    ax_prof.autoscale_view(scalex=True, scaley=False)
    # faint grid only for the shared y (already added below), keep profile axis clean
    ax_prof.grid(False)

    # ----- XY cross-section -----
    clevp = [x / 10**(-3) for x in clev]
    normp = BoundaryNorm(boundaries=clevp, ncolors=cmap.N, clip=True)
    contf1 = ax1.contourf(
        dsxy.xcr, dsxy.ycr, dsxy['pprime'],
        cmap=cmap, norm=normp, levels=clevp, extend='both'
    )
    ax1.contour(dsxy.xcr, dsxy.ycr, 100*dsxy["zcr"], colors='k',
                linestyles='dashed', levels=[0.2,0.4,0.6,0.8,1])
    ax1.set_ylabel("spanwise y / km")

    for ax in axes.flatten():
        ax.grid()
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())

    cbar0 = fig.colorbar(
        contf0, ax=axes[0,1], location='right', fraction=1, shrink=0.8,
        ticks=clev_l, label="w / m$\,$s$^{-1}$", pad=0.02, extend='both'
    )
    cbar0.formatter.set_powerlimits((0, 0))
    cbar0.formatter.set_useMathText(True)
     
    cbar1 = fig.colorbar(
        contf1, ax=axes[1,1], location='right', fraction=1, shrink=0.8,
        label="p' / Pa", pad=0.02, extend='both'
    )
    # cbar1.formatter.set_powerlimits((0, 0))
    # cbar1.formatter.set_useMathText(True)

    fig_title = 'slc_' + '{:03d}'.format(t) + '.png'
    fig.savefig(os.path.join(image_folder, fig_title), facecolor='w', edgecolor='w',
                format='png', dpi=300, bbox_inches='tight')
    plt.close(fig)

    plt_helper.show_progress(pbar['progress_counter'], pbar['lock'], pbar["stime"], pbar['ntasks'])
    sema.release()

if model == "pmap":
    trange = np.arange(0,300)
else:
    trange = np.arange(0,400)
    # trange = np.arange(0,90)

"""Parallel processing"""
progress_counter = mp.Manager().Value('i', 0)
lock = mp.Manager().Lock()
stime = time.time()
pbar = {"progress_counter": progress_counter, "lock": lock, "stime": stime, "ntasks": len(trange)}
ncpus = 200
ncpus = np.min([ncpus, pbar['ntasks']])
sema = mp.Semaphore(ncpus)

print(f"[i]  CPUs available: {mp.cpu_count()}")
print(f"[i]  CPUs for visualization: {ncpus}")

running_procs = []
for t in trange:
    for p in running_procs[:]:
        if not p.is_alive():
            p.join()
            running_procs.remove(p)
    sema.acquire()
    args = (t, fpath, pbar, sema)
    proc = mp.Process(target=plot_mw_sim, args=args)
    running_procs.append(proc)
    proc.start()

for proc in running_procs:
    proc.join()

[i]  CPUs available: 256
[i]  CPUs for visualization: 200
[p]  |#-------------------| Time: 2026-01-22 01:35:45 - Progress: 00.33% - Elapsed: 00:00:06 - ETA: 00:33:02 (hh:mm:ss)
[p]  |#-------------------| Time: 2026-01-22 01:35:48 - Progress: 05.00% - Elapsed: 00:00:09 - ETA: 00:03:06 (hh:mm:ss)
[p]  |##------------------| Time: 2026-01-22 01:35:50 - Progress: 10.00% - Elapsed: 00:00:11 - ETA: 00:01:44 (hh:mm:ss)
[p]  |###-----------------| Time: 2026-01-22 01:35:52 - Progress: 15.00% - Elapsed: 00:00:13 - ETA: 00:01:15 (hh:mm:ss)
[p]  |####----------------| Time: 2026-01-22 01:35:52 - Progress: 20.00% - Elapsed: 00:00:13 - ETA: 00:00:54 (hh:mm:ss)
[p]  |#####---------------| Time: 2026-01-22 01:35:53 - Progress: 25.00% - Elapsed: 00:00:14 - ETA: 00:00:43 (hh:mm:ss)
[p]  |######--------------| Time: 2026-01-22 01:35:56 - Progress: 30.00% - Elapsed: 00:00:17 - ETA: 00:00:41 (hh:mm:ss)
[p]  |#######-------------| Time: 2026-01-22 01:35:57 - Progress: 35.00% - Elapsed: 00:00:18 - ETA: 00

In [3]:
# sim = f"{model}_{flavor}_MW_010km"
# image_folder = f"../data/pmap-animations/{sim}"
plt_helper.create_animation(image_folder, f"anime_pmap_{sim}_{var}.mp4")

[i]  MP4 Video created successfully!
